In [ ]:
print(123)

In [ ]:
import pandas as pd

In [ ]:
url = "https://d37ci6vzurychx.cloudfront.net/trip-data/green_tripdata_2025-10.parquet"

In [ ]:
!uv pip install fsspec

In [ ]:
import fsspec
print(fsspec.__version__)

In [ ]:

columns = ['lpep_pickup_datetime', 'lpep_dropoff_datetime', 'PULocationID', 'DOLocationID', 'passenger_count', 'trip_distance', 'tip_amount', 'total_amount']
df = pd.read_parquet(url, columns=columns)


In [ ]:
df.head(5)

In [ ]:
df.info()

In [ ]:
df.count()

In [ ]:
from models_green import Ride, ride_from_row, ride_serializer

In [ ]:
ride = ride_from_row(df.iloc[0])
ride

In [ ]:
from kafka import KafkaProducer

server = 'localhost:9092'

producer = KafkaProducer(
    bootstrap_servers=[server],
    value_serializer=ride_serializer
)

In [ ]:
topic_name = 'green-trips'

In [ ]:
# producer.send(topic_name, value=ride)
# producer.flush()

In [ ]:
from time import time

t0 = time()

for _, row in df.iterrows():
    ride = ride_from_row(row)
    producer.send(topic_name, value=ride)
    #print(f"Sent: {ride}")
    #time.sleep(0.01)

producer.flush()

t1 = time()
print(f'took {(t1 - t0):.2f} seconds')